In [ ]:
!pip install -Uq ddgs
from ddgs import DDGS
from fastcore.all import *
from fastai.vision.all import *
from fastdownload import download_url
import time, random

def search_images(term, max_images=30, retries=5):
    print(f"Searching for '{term}'")
    for attempt in range(retries):
        try:
            with DDGS() as ddgs:
                results = ddgs.images(query=term, max_results=max_images)
                return L([r['image'] for r in results])
        except Exception as e:
            wait = 15 * (attempt + 1) + random.randint(5, 15)
            print(f"  {e.__class__.__name__}, waiting {wait}s (attempt {attempt+1}/{retries})")
            time.sleep(wait)
    print(f"  Failed after {retries} retries")
    return L([])


In [ ]:
urls = search_images('bird photos', max_images=10)
urls[0]


In [ ]:
download_url(urls[0], 'bird.jpg', show_progress=False)
im = Image.open('bird.jpg')
im.to_thumb(256,256)


In [ ]:
forest_urls = search_images('forest photos', max_images=10)
download_url(forest_urls[0], 'forest.jpg', show_progress=False)
Image.open('forest.jpg').to_thumb(256,256)

In [ ]:
searches = 'forest','bird'
path = Path('bird_or_not')
for o in searches:
    dest = (path/o)
    dest.mkdir(exist_ok=True, parents=True)
    download_images(dest, urls=search_images(f'{o} photo', max_images=50))
    time.sleep(15 + random.randint(5, 15))
    resize_images(path/o, max_size=400, dest=path/o)

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink)
len(failed)

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

dls.show_batch(max_n=6)

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

In [ ]:
is_bird,_,probs = learn.predict(PILImage.create('forest.jpg'))
print(f"This is a: {is_bird}.")
print(f"Probability it's a bird: {probs[0]:.4f}")